# 01 — NF-UNSW-NB15-v3 Data Audit

## Purpose

Audit the original NF-UNSW-NB15-v3 dataset before sampling,
preprocessing, splitting or model training.

## Research relevance

This audit establishes whether the dataset can support valid
development and evaluation samples for the COMPSCI 742 Assignment 2
group project.

## Input files

- `NF-UNSW-NB15-v3.csv` — main flow-level dataset
- `NetFlow_v3_Features.csv` — feature definitions
- `FurtherInformation.txt` — supplementary dataset information
- `manifest-sha1.txt` — file-integrity information

## Planned checks

1. Confirm that all required files exist.
2. Record file sizes and integrity information.
3. Inspect column names and data types.
4. Count binary labels and attack categories.
5. Identify missing and non-finite values.
6. assess exact and feature-level duplicate records.
7. Check consistency between `Label` and `Attack`.
8. Inspect timestamp range and class distribution over time.
9. Identify fields that may cause target leakage.
10. Document evidence needed for the later split decision.

## Scientific constraints

- The original files will not be modified.
- No rows will be deleted during this audit.
- No train/validation/test split will be created yet.
- `Label` and `Attack` are ground-truth fields and must not later be
  included in LLM or conventional-model inputs.
- Findings in this notebook are descriptive and do not yet constitute
  final experimental results.
  

In [4]:
# ============================================================
# Cell purpose:
# Import the standard Python libraries required for the initial
# data audit.
#
# Important:
# This cell does not read, modify or write any dataset.
# It only prepares the tools used in later audit steps.
# ============================================================

from pathlib import Path
# Path provides operating-system-independent file-path handling.
# It is safer and clearer than manually combining path strings.

import hashlib
# hashlib will be used to calculate SHA-1 hashes and verify that
# the downloaded files match the supplied dataset manifest.

import json
# json will later be used to save small, structured audit summaries.

import platform
import sys
# platform and sys record the Python and operating-system environment,
# which supports reproducibility.

import numpy as np
# NumPy will be used to identify non-finite numerical values,
# including positive infinity and negative infinity.

import pandas as pd
# pandas provides tabular data structures and chunked CSV reading.

In [5]:
# ============================================================
# Cell purpose:
# Define the project and input-file locations in one place.
#
# Input:
# The local project directory and the four downloaded source files.
#
# Output:
# Reusable Path objects for subsequent audit cells.
#
# Data mutation:
# None. This cell does not open or modify the source files.
# ============================================================

# The notebook is stored under:
# compsci742-rui-pilot/notebooks/
#
# Path.cwd() therefore points to the notebooks directory.
# `.parent` moves one level upward to the project root.
PROJECT_ROOT = Path.cwd().parent.resolve()

# Keep each dataset version in a separate, clearly named directory.
# This prevents NF-UNSW-NB15-v3 from being confused with the older
# UNSW-NB15 training/testing CSV files.
DATASET_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "nf_unsw_nb15_v3"
)

# Main flow-level dataset containing the feature columns and labels.
DATA_FILE = DATASET_DIR / "NF-UNSW-NB15-v3.csv"

# Dataset authors' definitions of the NetFlow-v3 features.
FEATURE_FILE = DATASET_DIR / "NetFlow_v3_Features.csv"

# Supplementary information distributed with the dataset.
INFO_FILE = DATASET_DIR / "FurtherInformation.txt"

# Author-supplied SHA-1 values used for file-integrity verification.
MANIFEST_FILE = DATASET_DIR / "manifest-sha1.txt"

print("Project root:", PROJECT_ROOT)
print("Dataset directory:", DATASET_DIR)

Project root: /Users/ruiwang/Developer/compsci742-rui-pilot
Dataset directory: /Users/ruiwang/Developer/compsci742-rui-pilot/data/raw/nf_unsw_nb15_v3


In [6]:
# ============================================================
# Cell purpose:
# Confirm that every required source file exists and record its
# local size before attempting to read the large dataset.
#
# Why this check comes first:
# A missing, incorrectly named or partially copied file could make
# later results misleading. Checking paths and file sizes provides
# an inexpensive first validation step.
#
# Data mutation:
# None. The files are inspected only through filesystem metadata.
# ============================================================

required_files = {
    "main_dataset": DATA_FILE,
    "feature_definitions": FEATURE_FILE,
    "further_information": INFO_FILE,
    "sha1_manifest": MANIFEST_FILE,
}

file_records = []

for logical_name, file_path in required_files.items():
    # Test whether the expected path exists on this computer.
    exists = file_path.exists()

    # Calculate file size only when the file exists.
    # Dividing by 1024**2 converts bytes into mebibytes (MiB).
    size_mib = (
        round(file_path.stat().st_size / (1024 ** 2), 3)
        if exists
        else None
    )

    file_records.append(
        {
            "logical_name": logical_name,
            "file_name": file_path.name,
            "exists": exists,
            "size_mib": size_mib,
            "absolute_path": str(file_path),
        }
    )

# Convert the audit records into a readable table.
file_check = pd.DataFrame(file_records)

# Stop immediately if any required file is missing.
# This prevents later cells from producing confusing errors.
assert file_check["exists"].all(), (
    "At least one required dataset file is missing. "
    "Review the paths shown in file_check."
)

file_check

,logical_name,file_name,exists,size_mib,absolute_path
0,main_dataset,NF-UNSW-NB15-v3.csv,True,550.614,/Users/ruiwang/Developer/compsci742-rui-pilot/...
1,feature_definitions,NetFlow_v3_Features.csv,True,0.003,/Users/ruiwang/Developer/compsci742-rui-pilot/...
2,further_information,FurtherInformation.txt,True,0.000,/Users/ruiwang/Developer/compsci742-rui-pilot/...
3,sha1_manifest,manifest-sha1.txt,True,0.000,/Users/ruiwang/Developer/compsci742-rui-pilot/...


## 1. Source-file integrity verification

The supplied SHA-1 manifest is used to determine whether the two CSV
files are identical to the files distributed by the dataset provider.

This step reads each file as binary data but does not parse or modify
the dataset. A matching SHA-1 value provides evidence that the local
copy was not corrupted or altered during download and copying.

In [7]:
# ============================================================
# Purpose:
# Compare the SHA-1 hashes of the local CSV files with the values
# supplied in manifest-sha1.txt.
#
# Why:
# A file may exist and have a plausible size while still being
# incomplete or corrupted. A matching cryptographic checksum gives
# stronger evidence that the local copy matches the distributed file.
#
# Input:
# - manifest-sha1.txt
# - NF-UNSW-NB15-v3.csv
# - NetFlow_v3_Features.csv
#
# Output:
# A table showing expected hash, calculated hash and match status.
#
# Data mutation:
# None. Files are opened in read-only binary mode.
# ============================================================

def calculate_sha1(file_path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    """
    Calculate the SHA-1 digest of a file without loading the complete
    file into memory.

    Parameters
    ----------
    file_path:
        Path to the file being verified.
    chunk_size:
        Number of bytes read per iteration. The default is 8 MiB.

    Returns
    -------
    str
        The lowercase hexadecimal SHA-1 digest.
    """
    sha1 = hashlib.sha1()

    # Reading in chunks keeps memory usage low, which matters because
    # the main CSV is approximately 550 MiB.
    with file_path.open("rb") as file_handle:
        while chunk := file_handle.read(chunk_size):
            sha1.update(chunk)

    return sha1.hexdigest()


# Read the provider-supplied manifest.
# Each line contains:
# <expected SHA-1 hash> <relative file path>
manifest_entries = {}

with MANIFEST_FILE.open("r", encoding="utf-8") as manifest_handle:
    for line in manifest_handle:
        line = line.strip()

        # Ignore empty lines, if any.
        if not line:
            continue

        expected_hash, relative_path = line.split(maxsplit=1)

        # The supplied paths begin with "data/" whereas our files are
        # stored in a version-specific local folder. Matching by the
        # final filename avoids depending on that packaging structure.
        file_name = Path(relative_path).name
        manifest_entries[file_name] = expected_hash.lower()


# Verify only the files listed in the supplied manifest.
files_to_verify = [
    DATA_FILE,
    FEATURE_FILE,
]

integrity_records = []

for file_path in files_to_verify:
    expected_hash = manifest_entries.get(file_path.name)
    calculated_hash = calculate_sha1(file_path)

    integrity_records.append(
        {
            "file_name": file_path.name,
            "expected_sha1": expected_hash,
            "calculated_sha1": calculated_hash,
            "sha1_match": calculated_hash == expected_hash,
        }
    )

integrity_check = pd.DataFrame(integrity_records)

# Stop the notebook if either CSV fails verification.
assert integrity_check["sha1_match"].all(), (
    "At least one CSV does not match the provider-supplied SHA-1 "
    "manifest. Do not continue until the affected file is replaced."
)

integrity_check

,file_name,expected_sha1,calculated_sha1,sha1_match
0,NF-UNSW-NB15-v3.csv,2cba4879ead17c51e3268a01c06af41186cd90e4,2cba4879ead17c51e3268a01c06af41186cd90e4,True
1,NetFlow_v3_Features.csv,d028597391217f78df3db31c1dbd96805203196f,d028597391217f78df3db31c1dbd96805203196f,True


## 2. Initial schema inspection

Only the header and first five records are loaded at this stage. This
provides an inexpensive check of the CSV structure without loading the
complete dataset into memory.

Data types inferred from five rows are preliminary and must not yet be
treated as the final column types.

In [8]:
# ============================================================
# Purpose:
# Read only the first five records to inspect the column names,
# ordering, example values and preliminary pandas data types.
#
# Why:
# The full CSV contains more than two million records. Reading a
# five-row preview is sufficient for an initial schema check and avoids
# unnecessary memory use.
#
# Important:
# - The `Label` and `Attack` columns are displayed here only for audit.
# - They must later be removed from all model inputs.
# - Data types inferred from five rows are provisional.
#
# Data mutation:
# None. The source CSV is opened in read-only mode.
# ============================================================

preview = pd.read_csv(
    DATA_FILE,
    nrows=5,
    low_memory=False,
)

print(f"Preview dimensions: {preview.shape[0]} rows × {preview.shape[1]} columns")
print(f"Number of detected columns: {len(preview.columns)}")

# Create a compact schema table that is easier to inspect than a
# horizontally wide five-row dataframe.
schema_preview = pd.DataFrame(
    {
        "column_position": range(1, len(preview.columns) + 1),
        "column_name": preview.columns,
        "provisional_dtype": preview.dtypes.astype(str).values,
    }
)

schema_preview

Preview dimensions: 5 rows × 55 columns
Number of detected columns: 55


,column_position,column_name,provisional_dtype
0,1,FLOW_START_MILLISECONDS,int64
1,2,FLOW_END_MILLISECONDS,int64
2,3,IPV4_SRC_ADDR,str
3,4,L4_SRC_PORT,int64
4,5,IPV4_DST_ADDR,str
5,6,L4_DST_PORT,int64
6,7,PROTOCOL,int64
7,8,L7_PROTO,float64
8,9,IN_BYTES,int64
9,10,IN_PKTS,int64


In [9]:
# Display the first five records.
# Jupyter may provide horizontal scrolling because the dataset contains
# many columns. This display is for visual inspection only.
preview

,FLOW_START_MILLISECONDS,FLOW_END_MILLISECONDS,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,...,SRC_TO_DST_IAT_MIN,SRC_TO_DST_IAT_MAX,SRC_TO_DST_IAT_AVG,SRC_TO_DST_IAT_STDDEV,DST_TO_SRC_IAT_MIN,DST_TO_SRC_IAT_MAX,DST_TO_SRC_IAT_AVG,DST_TO_SRC_IAT_STDDEV,Label,Attack
0,1424242193040,1424242193043,59.166.0.2,4894,149.171.126.3,53,17,5.0,146,2,...,0,0,0,0,0,0,0,0,0,Benign
1,1424242192744,1424242193079,59.166.0.4,52671,149.171.126.6,31992,6,11.0,4704,28,...,0,91,12,19,0,90,12,19,0,Benign
2,1424242190649,1424242193109,59.166.0.0,47290,149.171.126.9,6881,6,37.0,13662,238,...,0,1843,10,119,0,1843,5,88,0,Benign
3,1424242193145,1424242193146,59.166.0.8,43310,149.171.126.7,53,17,5.0,146,2,...,0,0,0,0,0,0,0,0,0,Benign
4,1424242193239,1424242193241,59.166.0.1,45870,149.171.126.1,53,17,5.0,130,2,...,0,0,0,0,0,0,0,0,0,Benign


## 3. Chunked full-dataset scan

The complete CSV is scanned in fixed-size chunks so that dataset-level
statistics can be calculated without loading all records into memory
at the same time.

This scan measures:

- total row count;
- binary-label distribution;
- attack-category distribution;
- missing-value counts;
- positive- and negative-infinity counts;
- earliest and latest flow timestamps; and
- basic consistency between `Label` and `Attack`.

No records are modified or removed.

In [10]:
# ============================================================
# Purpose:
# Scan the complete NF-UNSW-NB15-v3 CSV in manageable chunks and
# calculate dataset-level quality and distribution statistics.
#
# Why chunked reading:
# The CSV is approximately 550 MiB and contains more than two million
# records. Processing 100,000 rows at a time reduces peak memory use
# while still examining every record.
#
# Inputs:
# DATA_FILE — the verified original CSV.
#
# Outputs:
# - total_rows
# - label_counts
# - attack_counts
# - missing_counts
# - infinite_counts
# - minimum and maximum flow timestamps
# - label/category consistency counts
#
# Data mutation:
# None. The original CSV is read only. No output file is written.
# ============================================================

CHUNK_SIZE = 100_000

# Accumulators begin empty and are updated once per chunk.
total_rows = 0

label_counts = pd.Series(dtype="int64")
attack_counts = pd.Series(dtype="int64")

missing_counts = pd.Series(dtype="int64")
infinite_counts = pd.Series(dtype="int64")

minimum_start_ms = None
maximum_end_ms = None

# Consistency checks between the binary and categorical labels.
invalid_binary_label_count = 0
benign_label_category_mismatch_count = 0
attack_label_category_mismatch_count = 0

# pd.read_csv returns one DataFrame at a time when chunksize is set.
csv_chunks = pd.read_csv(
    DATA_FILE,
    chunksize=CHUNK_SIZE,
)

for chunk_number, chunk in enumerate(csv_chunks, start=1):
    # --------------------------------------------------------
    # 1. Count rows
    # --------------------------------------------------------
    total_rows += len(chunk)

    # --------------------------------------------------------
    # 2. Accumulate binary-label and attack-category counts
    # --------------------------------------------------------
    current_label_counts = chunk["Label"].value_counts(dropna=False)
    label_counts = label_counts.add(
        current_label_counts,
        fill_value=0,
    )

    current_attack_counts = chunk["Attack"].value_counts(dropna=False)
    attack_counts = attack_counts.add(
        current_attack_counts,
        fill_value=0,
    )

    # --------------------------------------------------------
    # 3. Count missing values in every column
    # --------------------------------------------------------
    current_missing_counts = chunk.isna().sum()
    missing_counts = missing_counts.add(
        current_missing_counts,
        fill_value=0,
    )

    # --------------------------------------------------------
    # 4. Count positive and negative infinity in numeric columns
    #
    # Missing values and infinity are different:
    # - NaN means missing or undefined data.
    # - inf/-inf are numerical values outside the finite range,
    #   often produced by division by zero in rate calculations.
    # --------------------------------------------------------
    numeric_chunk = chunk.select_dtypes(include=[np.number])

    current_infinite_counts = (
        np.isinf(numeric_chunk)
        .sum()
    )

    infinite_counts = infinite_counts.add(
        current_infinite_counts,
        fill_value=0,
    )

    # --------------------------------------------------------
    # 5. Track the complete timestamp range
    # --------------------------------------------------------
    current_minimum_start = chunk["FLOW_START_MILLISECONDS"].min()
    current_maximum_end = chunk["FLOW_END_MILLISECONDS"].max()

    if minimum_start_ms is None:
        minimum_start_ms = current_minimum_start
    else:
        minimum_start_ms = min(
            minimum_start_ms,
            current_minimum_start,
        )

    if maximum_end_ms is None:
        maximum_end_ms = current_maximum_end
    else:
        maximum_end_ms = max(
            maximum_end_ms,
            current_maximum_end,
        )

    # --------------------------------------------------------
    # 6. Check Label/Attack consistency
    #
    # Expected mapping:
    # Label == 0  -> Attack == "Benign"
    # Label == 1  -> Attack != "Benign"
    #
    # Category strings are stripped and converted to lowercase so
    # harmless capitalisation or whitespace differences do not create
    # false mismatches.
    # --------------------------------------------------------
    normalized_attack = (
        chunk["Attack"]
        .astype("string")
        .str.strip()
        .str.casefold()
    )

    valid_binary_label = chunk["Label"].isin([0, 1])
    invalid_binary_label_count += int((~valid_binary_label).sum())

    # Treat a missing Attack value paired with Label 0 as a mismatch.
    benign_category_is_not_benign = (
        normalized_attack.ne("benign")
        .fillna(True)
    )

    benign_label_category_mismatch_count += int(
        (
            chunk["Label"].eq(0)
            & benign_category_is_not_benign
        ).sum()
    )

    attack_label_category_mismatch_count += int(
        (
            chunk["Label"].eq(1)
            & normalized_attack.eq("benign").fillna(False)
        ).sum()
    )

    # Print occasional progress without flooding the notebook.
    if chunk_number == 1 or chunk_number % 5 == 0:
        print(
            f"Processed chunk {chunk_number:>2}: "
            f"{total_rows:,} cumulative rows"
        )

print("\nFull-dataset scan completed.")
print(f"Total rows processed: {total_rows:,}")

Processed chunk  1: 100,000 cumulative rows
Processed chunk  5: 500,000 cumulative rows
Processed chunk 10: 1,000,000 cumulative rows
Processed chunk 15: 1,500,000 cumulative rows
Processed chunk 20: 2,000,000 cumulative rows

Full-dataset scan completed.
Total rows processed: 2,365,424


In [11]:
# ============================================================
# Purpose:
# Convert the accumulated scan results into readable summary tables.
#
# Data mutation:
# None. This cell only formats statistics already held in memory.
# ============================================================

# Chunk-wise addition may convert integer counts to float.
# Convert them back to integers for clearer reporting.
label_counts = (
    label_counts
    .fillna(0)
    .astype("int64")
    .sort_index()
)

attack_counts = (
    attack_counts
    .fillna(0)
    .astype("int64")
    .sort_values(ascending=False)
)

missing_counts = (
    missing_counts
    .fillna(0)
    .astype("int64")
    .sort_values(ascending=False)
)

infinite_counts = (
    infinite_counts
    .fillna(0)
    .astype("int64")
    .sort_values(ascending=False)
)

# Convert millisecond Unix timestamps into readable UTC datetimes.
minimum_start_utc = pd.to_datetime(
    minimum_start_ms,
    unit="ms",
    utc=True,
)

maximum_end_utc = pd.to_datetime(
    maximum_end_ms,
    unit="ms",
    utc=True,
)

dataset_summary = pd.DataFrame(
    {
        "metric": [
            "Total records",
            "Total columns",
            "Earliest flow start (UTC)",
            "Latest flow end (UTC)",
            "Invalid binary labels",
            "Label 0 with non-Benign category",
            "Label 1 with Benign category",
        ],
        "value": [
            f"{total_rows:,}",
            len(preview.columns),
            str(minimum_start_utc),
            str(maximum_end_utc),
            f"{invalid_binary_label_count:,}",
            f"{benign_label_category_mismatch_count:,}",
            f"{attack_label_category_mismatch_count:,}",
        ],
    }
)

dataset_summary

,metric,value
0,Total records,"2,365,424"
1,Total columns,55
2,Earliest flow start (UTC),2015-01-22 11:49:36.907000+00:00
3,Latest flow end (UTC),2015-02-18 12:29:25.011000+00:00
4,Invalid binary labels,0
5,Label 0 with non-Benign category,0
6,Label 1 with Benign category,0


In [12]:
# ============================================================
# Purpose:
# Display binary-label and attack-category distributions with both
# counts and percentages.
#
# Why percentages matter:
# Accuracy can be misleading in an imbalanced dataset. If most records
# are Benign, a model can obtain high ordinary accuracy by predicting
# Benign too frequently.
# ============================================================

binary_distribution = pd.DataFrame(
    {
        "count": label_counts,
        "percentage": (
            label_counts / total_rows * 100
        ).round(4),
    }
)

binary_distribution.index.name = "Label"

category_distribution = pd.DataFrame(
    {
        "count": attack_counts,
        "percentage": (
            attack_counts / total_rows * 100
        ).round(4),
    }
)

category_distribution.index.name = "Attack"

print("Binary-label distribution:")
display(binary_distribution)

print("\nAttack-category distribution:")
display(category_distribution)

Binary-label distribution:


,count,percentage
Label,,
0,2237731,94.6017
1,127693,5.3983



Attack-category distribution:


,count,percentage
Attack,,
Benign,2237731,94.6017
Exploits,42748,1.8072
Fuzzers,33816,1.4296
Generic,19651,0.8308
Reconnaissance,17074,0.7218
DoS,5980,0.2528
Backdoor,4659,0.1970
Shellcode,2381,0.1007
Analysis,1226,0.0518


In [13]:
# ============================================================
# Purpose:
# Display only columns containing missing or infinite values.
#
# A zero-problem column is omitted from these tables to make the
# actual data-quality issues easier to identify.
# ============================================================

columns_with_missing_values = missing_counts[
    missing_counts > 0
].to_frame(name="missing_count")

if not columns_with_missing_values.empty:
    columns_with_missing_values["percentage"] = (
        columns_with_missing_values["missing_count"]
        / total_rows
        * 100
    ).round(4)

columns_with_infinite_values = infinite_counts[
    infinite_counts > 0
].to_frame(name="infinite_count")

if not columns_with_infinite_values.empty:
    columns_with_infinite_values["percentage"] = (
        columns_with_infinite_values["infinite_count"]
        / total_rows
        * 100
    ).round(4)

print("Columns containing missing values:")
display(columns_with_missing_values)

print("\nColumns containing positive or negative infinity:")
display(columns_with_infinite_values)

Columns containing missing values:


,missing_count,percentage
SRC_TO_DST_SECOND_BYTES,63425,2.6813



Columns containing positive or negative infinity:


,infinite_count,percentage
DST_TO_SRC_SECOND_BYTES,122493,5.1785
SRC_TO_DST_SECOND_BYTES,59068,2.4971


### Preliminary audit findings

The verified CSV contains 2,365,424 records and 55 columns. The
observed collection period extends from 22 January 2015 to 18 February
2015 in UTC.

The binary and categorical labels are internally consistent:

- no binary labels fall outside 0 and 1;
- no `Label = 0` record has a non-Benign category; and
- no `Label = 1` record has the Benign category.

The dataset is highly imbalanced. Benign traffic accounts for
94.6017% of all records, while the rarest attack category, Worms,
contains only 158 records.

The verified CSV contains 127,693 attack records. This differs by 54
records from the attack count of 127,639 stated on the dataset webpage.
The CSV-derived count is used here because it is consistent with both
the binary labels and the sum of the nine attack categories.

Non-finite values are concentrated in two directional byte-rate
features:

- `SRC_TO_DST_SECOND_BYTES` contains 63,425 missing values;
- `SRC_TO_DST_SECOND_BYTES` contains 59,068 infinite values; and
- `DST_TO_SRC_SECOND_BYTES` contains 122,493 infinite values.

The row-level overlap and category distribution of these non-finite
values must be examined before defining a cleaning or sampling rule.
No affected records have been removed at this stage.

In [14]:
# ============================================================
# Purpose:
# Determine how missing and infinite values overlap at row level
# and how frequently each quality pattern occurs by attack category.
#
# Why:
# Column-level counts cannot tell us how many unique records are
# affected. A single record may contain problems in both rate fields.
#
# This evidence is required before deciding whether to:
# - exclude affected records from the smoke pilot;
# - represent unavailable values explicitly; or
# - apply a documented imputation strategy.
#
# Data mutation:
# None. The original CSV is read in chunks and remains unchanged.
# ============================================================

QUALITY_COLUMNS = [
    "SRC_TO_DST_SECOND_BYTES",
    "DST_TO_SRC_SECOND_BYTES",
    "Label",
    "Attack",
]

quality_pattern_counts = pd.Series(dtype="int64")
affected_category_counts = pd.Series(dtype="int64")
eligible_category_counts = pd.Series(dtype="int64")

for chunk_number, chunk in enumerate(
    pd.read_csv(
        DATA_FILE,
        usecols=QUALITY_COLUMNS,
        chunksize=CHUNK_SIZE,
    ),
    start=1,
):
    # Identify the three non-finite conditions observed in the
    # full-dataset scan.
    src_is_missing = chunk["SRC_TO_DST_SECOND_BYTES"].isna()

    src_is_infinite = np.isinf(
        chunk["SRC_TO_DST_SECOND_BYTES"].to_numpy()
    )

    dst_is_infinite = np.isinf(
        chunk["DST_TO_SRC_SECOND_BYTES"].to_numpy()
    )

    # A record is considered affected if either directional rate field
    # is missing or infinite.
    any_nonfinite = (
        src_is_missing
        | src_is_infinite
        | dst_is_infinite
    )

    # Assign an interpretable quality-pattern label to every row.
    # np.select evaluates conditions in order.
    quality_pattern = np.select(
        [
            src_is_missing & dst_is_infinite,
            src_is_infinite & dst_is_infinite,
            src_is_missing,
            src_is_infinite,
            dst_is_infinite,
        ],
        [
            "src_missing_and_dst_infinite",
            "src_infinite_and_dst_infinite",
            "src_missing_only",
            "src_infinite_only",
            "dst_infinite_only",
        ],
        default="fully_finite",
    )

    current_pattern_counts = (
        pd.Series(quality_pattern)
        .value_counts()
    )

    quality_pattern_counts = quality_pattern_counts.add(
        current_pattern_counts,
        fill_value=0,
    )

    # Count affected and eligible records separately for every class.
    current_affected_counts = (
        chunk.loc[any_nonfinite, "Attack"]
        .value_counts(dropna=False)
    )

    affected_category_counts = affected_category_counts.add(
        current_affected_counts,
        fill_value=0,
    )

    current_eligible_counts = (
        chunk.loc[~any_nonfinite, "Attack"]
        .value_counts(dropna=False)
    )

    eligible_category_counts = eligible_category_counts.add(
        current_eligible_counts,
        fill_value=0,
    )

    if chunk_number == 1 or chunk_number % 5 == 0:
        print(
            f"Quality scan chunk {chunk_number:>2}: "
            f"{chunk_number * CHUNK_SIZE:,} rows reached"
        )

print("Row-level quality-pattern scan completed.")

Quality scan chunk  1: 100,000 rows reached
Quality scan chunk  5: 500,000 rows reached
Quality scan chunk 10: 1,000,000 rows reached
Quality scan chunk 15: 1,500,000 rows reached
Quality scan chunk 20: 2,000,000 rows reached
Row-level quality-pattern scan completed.


In [15]:
# ============================================================
# Purpose:
# Present row-level non-finite patterns and determine whether every
# category has enough fully finite records for the 200-row smoke pilot.
# ============================================================

quality_pattern_counts = (
    quality_pattern_counts
    .fillna(0)
    .astype("int64")
    .sort_values(ascending=False)
)

affected_category_counts = (
    affected_category_counts
    .fillna(0)
    .astype("int64")
)

eligible_category_counts = (
    eligible_category_counts
    .fillna(0)
    .astype("int64")
)

quality_pattern_table = pd.DataFrame(
    {
        "count": quality_pattern_counts,
        "percentage": (
            quality_pattern_counts / total_rows * 100
        ).round(4),
    }
)

category_quality_table = pd.DataFrame(
    {
        "total_count": attack_counts,
        "affected_count": affected_category_counts,
        "fully_finite_count": eligible_category_counts,
    }
).fillna(0).astype("int64")

category_quality_table["affected_percentage"] = (
    category_quality_table["affected_count"]
    / category_quality_table["total_count"]
    * 100
).round(4)

print("Row-level quality patterns:")
display(quality_pattern_table)

print("\nData quality by attack category:")
display(
    category_quality_table.sort_values(
        "total_count",
        ascending=False,
    )
)

# The first pilot requires at least 20 eligible records per class.
assert (
    category_quality_table["fully_finite_count"] >= 20
).all(), (
    "At least one category has fewer than 20 fully finite records. "
    "The initial sampling rule must be revised."
)

Row-level quality patterns:


,count,percentage
fully_finite,2242931,94.8215
src_missing_and_dst_infinite,63425,2.6813
src_infinite_and_dst_infinite,59068,2.4971



Data quality by attack category:


,total_count,affected_count,fully_finite_count,affected_percentage
Attack,,,,
Benign,2237731,86704,2151027,3.8746
Exploits,42748,3932,38816,9.1981
Fuzzers,33816,8228,25588,24.3317
Generic,19651,14891,4760,75.7773
Reconnaissance,17074,5783,11291,33.8702
DoS,5980,930,5050,15.5518
Backdoor,4659,1208,3451,25.9283
Shellcode,2381,795,1586,33.3893
Analysis,1226,0,1226,0.0000


## Preliminary data-audit findings

The provider-supplied CSV contains **2,365,424 network-flow records**
and **55 columns**. SHA-1 verification confirms that the two CSV files
match the checksums supplied with the dataset.

### Label integrity

- All values in `Label` are valid binary values.
- Every record with `Label = 0` is categorised as `Benign`.
- Every record with `Label = 1` has a non-Benign attack category.
- No label/category inconsistencies were detected.

### Class distribution

The dataset is strongly imbalanced. Benign traffic accounts for
approximately **94.60%** of all records. The smallest attack category,
`Worms`, contains only **158 records**.

The CSV contains **127,693 attack records**. This is 54 more than the
127,639 attack records reported on the dataset webpage. Because the
downloaded CSV passed the provider-supplied SHA-1 integrity check, the
difference is recorded as a source-documentation discrepancy rather
than treated as evidence of local file corruption.

### Missing and infinite values

A total of **122,493 records (5.1785%)** contain a non-finite value in
one or both directional second-byte features.

Two mutually exclusive patterns were observed:

1. 63,425 records contain a missing value in
   `SRC_TO_DST_SECOND_BYTES` and an infinite value in
   `DST_TO_SRC_SECOND_BYTES`.
2. 59,068 records contain infinite values in both
   `SRC_TO_DST_SECOND_BYTES` and `DST_TO_SRC_SECOND_BYTES`.

No records contain only an isolated source-side or destination-side
non-finite value.

### Implication for sampling and preprocessing

Non-finite values are not distributed uniformly across attack
categories. For example, approximately 3.87% of Benign records are
affected, compared with approximately 75.78% of Generic records.

Consequently, removing every affected record from the final experiment
could introduce class-dependent selection bias. The treatment of these
values must therefore be specified as part of the preprocessing
protocol and applied identically across all experimental conditions.

For the initial development smoke test only, sampling may be restricted
to fully finite records so that the end-to-end pipeline can be validated.
Results from that restricted sample must not be reported as final
generalisation performance.